### imports and pandas settings

In [11]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [12]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [13]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [14]:
batch_data = CF_OUTPUTS / "base_vs_thresholds" / "base_rf_midthres_2026-05-06" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [15]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [16]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [17]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [18]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.47,,,,,
1,0,cf_1,,,,,,,16.7521,,,0.9451,1,False,0.0633,0.0733
2,0,cf_2,,,,7,,,18.9745,,,0.8754,2,False,0.0633,0.07
3,0,cf_3,1,,,,,,18.9619,,,0.8758,2,False,0.0633,0.15
4,0,cf_4,2,,,,,,18.8358,,,0.8797,2,False,0.0633,0.1433
5,0,cf_5,,,,7,,,18.4476,,,0.8919,2,False,0.0633,0.0567
6,0,cf_6,,,,,,,15.0,,,1.0,1,False,0.0633,0.17
7,0,cf_7,1,,,,,,17.516,,,0.9211,2,False,0.0633,0.14
8,0,cf_8,,,,6,,,17.0974,,,0.9342,2,False,0.0633,0.2067
9,0,cf_9,3,,,,,,16.7519,,,0.9451,2,False,0.0633,0.09


### Filtering Valid CFs

In [19]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.47,,,,,
1,1,xin,3,4,1,2,3,0,22.3757,0,2.54,,,,,
9,1,cf_3,,,2,,1,,22.3757,,,0.8796,3,True,0.1467,0.0233
10,1,cf_5,,,,3,1,,22.3757,,,0.8796,3,True,0.1467,0.0
11,1,cf_7,,,,,1,,22.3757,1,,0.8796,3,True,0.1467,0.05
12,1,cf_10,2,,,,1,,22.3757,,,0.8796,3,True,0.1467,0.01
2,2,xin,5,3,1,1,4,0,22.694,7,2.62,,,,,
13,2,cf_4,3,,,,2,,22.68,,,0.8755,3,True,0.1567,0.0133
14,2,cf_5,,,,4,2,,22.6757,,,0.8756,3,True,0.1567,0.0633
15,2,cf_8,,,,6,1,,22.6757,,,0.8756,3,True,0.1567,0.0633


### select one CF

In [20]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.47,,,,,
1,1,xin,3,4,1,2,3,0,22.3757,0,2.54,,,,,
9,1,cf_5,,,,3,1,,22.3757,,,0.8796,3,True,0.1467,0.0
2,2,xin,5,3,1,1,4,0,22.694,7,2.62,,,,,
10,2,cf_5,,,,4,2,,22.6757,,,0.8756,3,True,0.1567,0.0633
3,3,xin,3,3,6,6,2,0,24.3418,1,2.83,,,,,
11,3,cf_1,,,,,,,24.3375,,,0.8781,1,True,0.02,0.01
4,4,xin,5,4,2,7,1,0,21.2585,3,2.94,,,,,
12,4,cf_6,,3,,,,,21.2585,,,0.9868,2,True,0.0267,0.0133
5,5,xin,2,2,1,2,4,0,27.9904,0,2.68,,,,,


In [21]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
No recommendations available.

=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_3 ---
Predicted risk: 0.0233
Valid: True
Changes:
  cgtsmok: 1 → 2
  slprl: 3 → 1

--- cf_5 ---
Predicted risk: 0.0
Valid: True
Changes:
  alcfreq: 2 → 3
  slprl: 3 → 1

--- cf_7 ---
Predicted risk: 0.05
Valid: True
Changes:
  slprl: 3 → 1
  dosprt: 0 → 1

--- cf_10 ---
Predicted risk: 0.01
Valid: True
Changes:
  etfruit: 3 → 2
  slprl: 3 → 1


=== Query index '2' ===
Task / Target: hltprhc
Query instance index: 2

Original instance:
  etfruit: 5
  eatveg: 3
  cgtsmok: 1
  alcfreq: 1
  slprl: 4
  paccnois: 0
  bmi: 22.694
  dosprt: 7


=== Counterfactuals ===

--- cf_4 ---
Predicted risk: 0.0133
Valid: True
Changes:
  etfruit: 5 → 3
  slprl: 4 → 2
  bmi: 22.694 → 22.68

--- cf_5 ---
Predicted risk: 0.0633
V